In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
import os

CATALOG = "uc13"
#COMPANY_NAME = dbutils.secrets.get("uc13", "sp_company_name")
COMPANY_NAME = "Elder Care"
TABLE_PROFILE = f"{CATALOG}.classification.company_profile"
EMBEDDING_ENDPOINT = "databricks-bge-large-en"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

from pathlib import Path

def get_current_path():
    try:
        notebook_path = (
            dbutils.notebook.entry_point
            .getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        return Path("/Workspace") / notebook_path.lstrip("/")
    except Exception:
        return Path(os.getcwd())

def find_repo_root(marker="agents"):
    current_path = get_current_path()

    if current_path.is_file():
        current_path = current_path.parent

    for path in [current_path, *current_path.parents]:
        if (path / marker).exists():
            return str(path)

    raise RuntimeError(f"No se encontró una carpeta padre que contenga '{marker}'")

repo_root = find_repo_root()

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("repo_root:", repo_root)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_PROFILE} (
  company_name        STRING,
  industry_vertical   STRING,
  sub_vertical        STRING,
  benchmark_overlay   STRING,
  business_model      STRING,
  revenue_model       STRING,
  geographies         STRING,
  key_services        STRING,
  raw_llm_output      STRING,
  profiled_at         TIMESTAMP
) USING DELTA
""")

print(f"Company: {COMPANY_NAME}")
print("Setup done")

In [0]:
from agents.shared.retrieval import semantic_search

def search(query, top_k=10, workstream_filter=None, tier_filter=None):
    return semantic_search(
        query=query,
        spark=spark,
        top_k=top_k,
        workstream_filter=workstream_filter,
        tier_filter=tier_filter
    )

print("Retrieval module loaded")

In [0]:
# Each query targets a specific aspect of the company profile
# workstream_filter focuses retrieval on the most relevant document type
# tier_filter=1 ensures we only use the highest quality documents

profiling_queries = {
    "business_description": (
        "company overview what does this company do business description services offered",
        "ma"
    ),
    "industry_vertical": (
        "industry sector healthcare services home care hospice SaaS manufacturing",
        "ma"
    ),
    "revenue_model": (
        "revenue model recurring subscription retainer project based contracts revenue streams",
        None
    ),
    "geographies": (
        "states regions locations where company operates geographic markets",
        None
    ),
    "key_services": (
        "main services products offerings caregiver staffing clinical",
        None
    ),
    "customers": (
        "customers clients payor mix Medicare Medicaid commercial concentration",
        None
    ),
    "financials_summary": (
        "revenue EBITDA gross margin growth rate financial performance",
        "financial"
    ),
}

retrieved_context = {}

for aspect, (query, ws_filter) in profiling_queries.items():
    chunks = search(
        query=query,
        top_k=5,
        tier_filter=1,
        workstream_filter=ws_filter
    )
    if chunks:
        context = "\n---\n".join([
            f"[Source: {c.file_name} | Section: {c.section_header or 'N/A'}]\n{c.chunk_text[:800]}"
            for c in chunks
        ])
        retrieved_context[aspect] = context
        print(f"✓ {aspect}: {len(chunks)} chunks from {len(set(c.file_name for c in chunks))} files")
    else:
        retrieved_context[aspect] = "No relevant information found"
        print(f"✗ {aspect}: no chunks found")

In [0]:
print(retrieved_context['business_description'])

In [0]:
print(retrieved_context['geographies'])

In [0]:
import json
import re
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

# Build context — cap each aspect at 800 chars so all 7 fit in the window
context_parts = []
for aspect, context in retrieved_context.items():
    context_parts.append(f"## {aspect.upper()}\n{context[:800]}")

full_context = "\n\n========\n\n".join(context_parts)
print(f"Total context length: {len(full_context)}")

json_template = """{
  "industry_vertical": "one of: healthcare_services, b2b_saas, professional_services, industrial_manufacturing, consumer_dtc, other",
  "sub_vertical": "specific sub-sector e.g. home_care, hospice, behavioral_health, staffing, dental",
  "benchmark_overlay": "one of: healthcare, saas, services, industrial, consumer",
  "business_model": "2-3 sentence description of what the company does and how it makes money",
  "revenue_model": "one of: recurring_subscription, repeat_services, project_based, transactional, hybrid",
  "geographies": "comma-separated list of states or regions where company operates",
  "key_services": "comma-separated list of main services or products"
}"""

prompt = (
    "You are a private equity analyst performing company profiling from due diligence documents.\n"
    f"The company being analyzed is: {COMPANY_NAME}\n"
    "The documents below may include a CIM, financial models, contracts, or operational reports.\n"
    "Use whatever is available to build the most accurate profile possible.\n"
    "If information for a field is not found in the documents, use null for that field.\n\n"
    "IMPORTANT EXTRACTION HINTS:\n"
    "- geographies: look for states, cities, counties where the company operates or plans to expand\n"
    "- revenue_model: home care companies typically bill per hour or per visit — this is repeat_services\n"
    "- benchmark_overlay: home care = healthcare\n\n"
    "Return ONLY a valid JSON object, no markdown, no explanation:\n"
    + json_template
    + "\n\nDue Diligence Documents:\n"
    + full_context[:10000]
)

response = client.predict(
    endpoint=LLM_ENDPOINT,
    inputs={
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1000,
        "temperature": 0.0
    }
)

text = response["choices"][0]["message"]["content"].strip()
text = re.sub(r'```json\s*|\s*```', '', text).strip()

try:
    profile = json.loads(text)
    print("✓ Company profile extracted:\n")
    for k, v in profile.items():
        print(f"  {k:<25} {v}")
except Exception as e:
    print(f"Parse error: {e}")
    print(f"Raw: {text[:500]}")
    profile = None

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from datetime import datetime

if profile:
    schema = StructType([
        StructField("company_name",      StringType(), True),
        StructField("industry_vertical", StringType(), True),
        StructField("sub_vertical",      StringType(), True),
        StructField("benchmark_overlay", StringType(), True),
        StructField("business_model",    StringType(), True),
        StructField("revenue_model",     StringType(), True),
        StructField("geographies",       StringType(), True),
        StructField("key_services",      StringType(), True),
        StructField("raw_llm_output",    StringType(), True),
        StructField("profiled_at",       TimestampType(), True),
    ])

    row = Row(
        company_name=COMPANY_NAME,
        industry_vertical=profile.get("industry_vertical"),
        sub_vertical=profile.get("sub_vertical"),
        benchmark_overlay=profile.get("benchmark_overlay"),
        business_model=profile.get("business_model"),
        revenue_model=profile.get("revenue_model"),
        geographies=profile.get("geographies"),
        key_services=profile.get("key_services"),
        raw_llm_output=text,
        profiled_at=datetime.now()
    )

    df = spark.createDataFrame([row], schema=schema)
    df.write.mode("overwrite").saveAsTable(TABLE_PROFILE)
    print(f"✓ Profile saved to {TABLE_PROFILE}")
else:
    print("No profile to save — check parse error above")

In [0]:
%sql
SELECT
  company_name,
  industry_vertical,
  sub_vertical,
  benchmark_overlay,
  revenue_model,
  geographies,
  key_services
FROM uc13.classification.company_profile

In [0]:
%sql
TRUNCATE TABLE uc13.classification.company_profile;
SELECT 'Cleared' as status;

In [0]:
%sql
-- First see exactly what's stored
SELECT company_name, length(company_name) as len
FROM uc13.classification.company_profile;